# Usar service.py directamente (sin FastAPI)

Este notebook llama las funciones de `src/service.py` sin pasar por el servidor HTTP.

**Funciones disponibles:**
- `run_analysis(data, n_dims)` — estima los parámetros latentes a partir de menciones.
- `generate_data(outlets, subjects, amount_of_mentions)` — genera datos sintéticos a partir de parámetros conocidos.

In [1]:
import sys
import os

# Agrega la raíz del proyecto al path para poder importar src/
sys.path.insert(0, os.path.abspath(".."))

In [2]:
from src.service import run_analysis, generate_data
from src.schemas import Mention, AnalysisInput, OutletScore, SubjectScore

---
## 1. Analizar datos (`run_analysis`)

Definí el JSON con las menciones y corré la estimación.

In [3]:
# --- Editá este bloque con tus datos ---
input_json = {
    "n_dimensions": 1,
    "data": [
        {"outlet": "A", "subject": "X", "mention_type": "positive",  "amount_of_mentions": 12},
        {"outlet": "A", "subject": "X", "mention_type": "negative",  "amount_of_mentions": 3},
        {"outlet": "A", "subject": "Y", "mention_type": "negative",  "amount_of_mentions": 5},
        {"outlet": "A", "subject": "Y", "mention_type": "neutral",   "amount_of_mentions": 2},
        {"outlet": "B", "subject": "X", "mention_type": "neutral",   "amount_of_mentions": 8},
        {"outlet": "B", "subject": "X", "mention_type": "positive",  "amount_of_mentions": 2},
        {"outlet": "B", "subject": "Y", "mention_type": "positive",  "amount_of_mentions": 7},
        {"outlet": "C", "subject": "X", "mention_type": "negative",  "amount_of_mentions": 4},
        {"outlet": "C", "subject": "X", "mention_type": "positive",  "amount_of_mentions": 6},
    ]
}
# ---------------------------------------

# También podés cargar desde un archivo:
# import json
# input_json = json.load(open("../data/input.json"))

In [4]:
# Validar y parsear con Pydantic
analysis_input = AnalysisInput(**input_json)

# Correr la estimación
result = run_analysis(analysis_input.data, n_dims=analysis_input.n_dimensions)
result

AnalysisOutput(outlets=[OutletScore(outlet='A', z=[1.0353505921833284]), OutletScore(outlet='B', z=[-1.1285337673804043]), OutletScore(outlet='C', z=[-0.3343367536392774])], subjects=[SubjectScore(subject='X', a=[0.3428149851637815], b=0.5767645423140676), SubjectScore(subject='Y', a=[-1.5295749260270768], b=0.4096990678881777)], loss=41.94498307426818, bic=111.13270823531074)

In [5]:
# Ver resultados por medio
for outlet in result.outlets:
    print(f"{outlet.outlet:30s}  z = {outlet.z}")

A                               z = [1.0353505921833284]
B                               z = [-1.1285337673804043]
C                               z = [-0.3343367536392774]


In [6]:
# Ver resultados por sujeto
for subject in result.subjects:
    print(f"{subject.subject:30s}  a = {subject.a}  b = {subject.b:.4f}")

X                               a = [0.3428149851637815]  b = 0.5768
Y                               a = [-1.5295749260270768]  b = 0.4097


In [7]:
print(f"Loss (neg-log-likelihood): {result.loss:.4f}")
print(f"BIC:                       {result.bic:.4f}")

Loss (neg-log-likelihood): 41.9450
BIC:                       111.1327


---
## 2. Generar datos sintéticos (`generate_data`)

Pasale parámetros conocidos (por ejemplo, la salida anterior) y generá menciones simuladas.

In [8]:
# Usar la salida de run_analysis como parámetros de entrada
generated = generate_data(
    outlets=result.outlets,
    subjects=result.subjects,
    amount_of_mentions=100,
)
generated

AnalysisInput(data=[Mention(outlet='A', subject='X', mention_type='negative', amount_of_mentions=5), Mention(outlet='A', subject='X', mention_type='neutral', amount_of_mentions=26), Mention(outlet='A', subject='X', mention_type='positive', amount_of_mentions=69), Mention(outlet='A', subject='Y', mention_type='negative', amount_of_mentions=64), Mention(outlet='A', subject='Y', mention_type='neutral', amount_of_mentions=27), Mention(outlet='A', subject='Y', mention_type='positive', amount_of_mentions=9), Mention(outlet='B', subject='X', mention_type='negative', amount_of_mentions=29), Mention(outlet='B', subject='X', mention_type='neutral', amount_of_mentions=30), Mention(outlet='B', subject='X', mention_type='positive', amount_of_mentions=41), Mention(outlet='B', subject='Y', mention_type='negative', amount_of_mentions=1), Mention(outlet='B', subject='Y', mention_type='neutral', amount_of_mentions=10), Mention(outlet='B', subject='Y', mention_type='positive', amount_of_mentions=89), Men

In [9]:
# Ver las primeras menciones generadas
for mention in generated.data[:12]:
    print(mention)

outlet='A' subject='X' mention_type='negative' amount_of_mentions=5
outlet='A' subject='X' mention_type='neutral' amount_of_mentions=26
outlet='A' subject='X' mention_type='positive' amount_of_mentions=69
outlet='A' subject='Y' mention_type='negative' amount_of_mentions=64
outlet='A' subject='Y' mention_type='neutral' amount_of_mentions=27
outlet='A' subject='Y' mention_type='positive' amount_of_mentions=9
outlet='B' subject='X' mention_type='negative' amount_of_mentions=29
outlet='B' subject='X' mention_type='neutral' amount_of_mentions=30
outlet='B' subject='X' mention_type='positive' amount_of_mentions=41
outlet='B' subject='Y' mention_type='negative' amount_of_mentions=1
outlet='B' subject='Y' mention_type='neutral' amount_of_mentions=10
outlet='B' subject='Y' mention_type='positive' amount_of_mentions=89


---
## 3. Ciclo completo: generar → estimar

Generá datos desde parámetros conocidos y verificá que la estimación los recupera.

In [10]:
# Parámetros conocidos de verdad
true_outlets = [
    OutletScore(outlet="Medio_izquierda", z=[-1.5]),
    OutletScore(outlet="Medio_centro",    z=[0.0]),
    OutletScore(outlet="Medio_derecha",   z=[1.5]),
]
true_subjects = [
    SubjectScore(subject="Candidato_A", a=[1.0], b=0.0),
    SubjectScore(subject="Candidato_B", a=[-1.0], b=0.0),
]

synthetic = generate_data(true_outlets, true_subjects, amount_of_mentions=200)
print(f"Menciones generadas: {len(synthetic.data)}")

Menciones generadas: 18


In [11]:
estimated = run_analysis(synthetic.data, n_dims=synthetic.n_dimensions)

print("Outlets estimados:")
for o in estimated.outlets:
    print(f"  {o.outlet:25s}  z = {[f'{v:.3f}' for v in o.z]}")

print("\nSujetos estimados:")
for s in estimated.subjects:
    print(f"  {s.subject:25s}  a = {[f'{v:.3f}' for v in s.a]}  b = {s.b:.3f}")

Outlets estimados:
  Medio_centro               z = ['-0.000']
  Medio_derecha              z = ['1.061']
  Medio_izquierda            z = ['-1.151']

Sujetos estimados:
  Candidato_A                a = ['1.264']  b = -0.001
  Candidato_B                a = ['-1.200']  b = 0.027
